# Data Exploration

This notebook explores the dataset structure, visualizes sample images, and analyzes class distribution.

In [ ]:
# Import necessary libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import pandas as pd

# Add src directory to path
import sys
sys.path.append('../src')

# Import project modules
from config import TRAIN_DIR, TEST_DIR, IMAGE_SIZE
from preprocess import load_images_from_directory, preprocess_images

## Dataset Overview

In [ ]:
# Check dataset structure
print("Dataset Structure:")
print(f"Training directory: {TRAIN_DIR}")
print(f"Test directory: {TEST_DIR}")

# List classes in training data
if os.path.exists(TRAIN_DIR):
    train_classes = os.listdir(TRAIN_DIR)
    print(f"\nTraining classes: {train_classes}")
    
    # Count images per class
    for class_name in train_classes:
        class_dir = os.path.join(TRAIN_DIR, class_name)
        if os.path.isdir(class_dir):
            num_images = len([f for f in os.listdir(class_dir) 
                            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            print(f"  {class_name}: {num_images} images")

# Check test data
if os.path.exists(TEST_DIR):
    test_classes = os.listdir(TEST_DIR)
    print(f"\nTest classes: {test_classes}")
    
    for class_name in test_classes:
        class_dir = os.path.join(TEST_DIR, class_name)
        if os.path.isdir(class_dir):
            num_images = len([f for f in os.listdir(class_dir) 
                            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            print(f"  {class_name}: {num_images} images")

## Load Sample Images

In [ ]:
# Load sample images from each class
train_images, train_labels = load_images_from_directory(TRAIN_DIR)

print(f"Loaded {len(train_images)} training images")
print(f"Image shape: {train_images[0].shape}")
print(f"Labels: {set(train_labels)}")

## Visualize Sample Images

In [ ]:
# Display sample images from each class
unique_labels = list(set(train_labels))
n_classes = len(unique_labels)

fig, axes = plt.subplots(1, n_classes, figsize=(15, 5))
if n_classes == 1:
    axes = [axes]

for i, label in enumerate(unique_labels):
    # Find first image of this class
    idx = train_labels.index(label)
    img = train_images[idx]
    
    # Convert BGR to RGB for display
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(img_rgb)
    axes[i].set_title(f'Class: {label}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Class Distribution Analysis

In [ ]:
# Analyze class distribution
label_counts = Counter(train_labels)

# Create bar plot
plt.figure(figsize=(10, 6))
labels = list(label_counts.keys())
counts = list(label_counts.values())

bars = plt.bar(labels, counts, color=['skyblue', 'lightcoral'])
plt.title('Class Distribution in Training Data')
plt.xlabel('Class')
plt.ylabel('Number of Images')

# Add value labels on bars
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(count), ha='center', va='bottom')

plt.show()

print(f"Total images: {sum(counts)}")
print(f"Number of classes: {len(labels)}")
print(f"Images per class: {dict(label_counts)}")

## Image Properties Analysis

In [ ]:
# Analyze image properties
image_sizes = []
aspect_ratios = []

for img in train_images[:100]:  # Sample first 100 images
    height, width = img.shape[:2]
    image_sizes.append((width, height))
    aspect_ratios.append(width / height)

# Convert to arrays
widths = [size[0] for size in image_sizes]
heights = [size[1] for size in image_sizes]

# Plot image size distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Width distribution
ax1.hist(widths, bins=20, alpha=0.7, color='skyblue')
ax1.set_title('Image Width Distribution')
ax1.set_xlabel('Width (pixels)')
ax1.set_ylabel('Frequency')

# Height distribution
ax2.hist(heights, bins=20, alpha=0.7, color='lightcoral')
ax2.set_title('Image Height Distribution')
ax2.set_xlabel('Height (pixels)')
ax2.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Aspect ratio distribution
plt.figure(figsize=(10, 6))
plt.hist(aspect_ratios, bins=20, alpha=0.7, color='lightgreen')
plt.title('Aspect Ratio Distribution')
plt.xlabel('Aspect Ratio (Width/Height)')
plt.ylabel('Frequency')
plt.show()

print(f"Image size statistics:")
print(f"  Width: min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.1f}")
print(f"  Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.1f}")
print(f"  Aspect ratio: min={min(aspect_ratios):.2f}, max={max(aspect_ratios):.2f}, mean={np.mean(aspect_ratios):.2f}")

## Color Channel Analysis

In [ ]:
# Analyze color channels for sample images
sample_images = train_images[:5]  # Take first 5 images

fig, axes = plt.subplots(5, 4, figsize=(20, 25))

for i, img in enumerate(sample_images):
    # Convert BGR to RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Original image
    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_title(f'Original - Class: {train_labels[i]}')
    axes[i, 0].axis('off')
    
    # Red channel
    axes[i, 1].imshow(img_rgb[:, :, 0], cmap='Reds')
    axes[i, 1].set_title('Red Channel')
    axes[i, 1].axis('off')
    
    # Green channel
    axes[i, 2].imshow(img_rgb[:, :, 1], cmap='Greens')
    axes[i, 2].set_title('Green Channel')
    axes[i, 2].axis('off')
    
    # Blue channel
    axes[i, 3].imshow(img_rgb[:, :, 2], cmap='Blues')
    axes[i, 3].set_title('Blue Channel')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

## Preprocessing Effects

In [ ]:
# Show effects of preprocessing
sample_img = train_images[0]

# Original image
original_rgb = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)

# Resized image
resized = cv2.resize(sample_img, IMAGE_SIZE)
resized_rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)

# Normalized image
normalized = resized.astype(np.float32) / 255.0

# Grayscale
grayscale = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)

# Plot preprocessing steps
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(original_rgb)
axes[0, 0].set_title(f'Original ({sample_img.shape[:2]})')
axes[0, 0].axis('off')

axes[0, 1].imshow(resized_rgb)
axes[0, 1].set_title(f'Resized ({IMAGE_SIZE})')
axes[0, 1].axis('off')

axes[1, 0].imshow(normalized)
axes[1, 0].set_title('Normalized [0,1]')
axes[1, 0].axis('off')

axes[1, 1].imshow(grayscale, cmap='gray')
axes[1, 1].set_title('Grayscale')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
# Create summary statistics
summary_stats = {
    'total_images': len(train_images),
    'num_classes': len(set(train_labels)),
    'class_distribution': dict(Counter(train_labels)),
    'image_shape': train_images[0].shape,
    'target_size': IMAGE_SIZE,
    'color_channels': 3
}

print("Dataset Summary Statistics:")
print("=" * 40)
for key, value in summary_stats.items():
    print(f"{key}: {value}")

# Check for class imbalance
max_count = max(summary_stats['class_distribution'].values())
min_count = min(summary_stats['class_distribution'].values())
imbalance_ratio = max_count / min_count

print(f"\nClass Imbalance Analysis:")
print(f"  Max class count: {max_count}")
print(f"  Min class count: {min_count}")
print(f"  Imbalance ratio: {imbalance_ratio:.2f}")

if imbalance_ratio > 2:
    print("  ⚠️  Dataset is imbalanced. Consider using balancing techniques.")
else:
    print("  ✅ Dataset is reasonably balanced.")